# Upload CICIoT2023 to Google Drive

Local source: `/Volumes/LocalDrive1/Dataset/Security/CICIoT2023/raw/wataiData`

Drive destination: `gdrive:projects/Dataset/Security/CICIoT2023`

In [1]:
from pathlib import Path
import os
import shutil
import subprocess

DATASET_NAME = "CICIoT2023"
LOCAL_SOURCE = Path("/Volumes/LocalDrive1/Dataset/Security/CICIoT2023/raw/wataiData")
DRIVE_DESTINATION = "gdrive:projects/Dataset/Security/CICIoT2023"
EXPECTED_MIN_CSV_COUNT = 169
EXCLUDES = ["._*", ".DS_Store", "*.part"]

def find_rclone() -> str:
    env_path = os.environ.get("RCLONE_BIN")
    if env_path and Path(env_path).exists():
        return env_path
    found = shutil.which("rclone")
    if found:
        return found
    candidates = ["/opt/homebrew/bin/rclone", "/usr/local/bin/rclone", "/opt/local/bin/rclone"]
    candidates.extend(str(p) for p in Path("/opt/homebrew/Cellar/rclone").glob("*/bin/rclone"))
    for candidate in candidates:
        if Path(candidate).exists():
            return candidate
    for shell in ["/bin/zsh", "/usr/bin/zsh", "/bin/bash", "/usr/bin/bash", "/bin/sh", "/usr/bin/sh"]:
        if not Path(shell).exists():
            continue
        try:
            shell_found = subprocess.run([shell, "-lc", "command -v rclone"], capture_output=True, text=True)
        except OSError:
            continue
        if shell_found.returncode == 0 and shell_found.stdout.strip():
            return shell_found.stdout.strip()
    checked = ", ".join(candidates)
    raise RuntimeError(
        "rclone is not available in this runtime. Run this on the local Mac runtime with rclone configured. "
        "If 01 printed a working path, set it here first: os.environ['RCLONE_BIN'] = '/that/path'. "
        f"Checked: {checked}"
    )


if not LOCAL_SOURCE.exists():
    raise FileNotFoundError(
        f"Missing local source: {LOCAL_SOURCE}. This upload notebook must run on the local Mac runtime "
        "that can access /Volumes/LocalDrive1, not a Colab/cloud runtime."
    )

RCLONE = find_rclone()
print("rclone:", RCLONE)

csv_files = [p for p in LOCAL_SOURCE.rglob("*.csv") if p.is_file() and not p.name.startswith("._")]
print("Dataset:", DATASET_NAME)
print("Local source:", LOCAL_SOURCE)
print("Source is symlink:", LOCAL_SOURCE.is_symlink())
print("CSV count:", len(csv_files))
print("Drive destination:", DRIVE_DESTINATION)
if len(csv_files) < EXPECTED_MIN_CSV_COUNT:
    raise RuntimeError(f"Expected at least {EXPECTED_MIN_CSV_COUNT} CSV files, found {len(csv_files)}")

rclone: /opt/homebrew/bin/rclone
Dataset: CICIoT2023
Local source: /Volumes/LocalDrive1/Dataset/Security/CICIoT2023/raw/wataiData
Source is symlink: True
CSV count: 169
Drive destination: gdrive:projects/Dataset/Security/CICIoT2023


In [2]:
cmd = [
    RCLONE, "copy", str(LOCAL_SOURCE), DRIVE_DESTINATION,
    "--copy-links",
    "--progress",
    "--transfers", "4",
    "--checkers", "8",
    "--drive-chunk-size", "256M",
    "--stats", "30s",
]
for pattern in EXCLUDES:
    cmd.extend(["--exclude", pattern])

print(" ".join(cmd))
subprocess.check_call(cmd)

/opt/homebrew/bin/rclone copy /Volumes/LocalDrive1/Dataset/Security/CICIoT2023/raw/wataiData gdrive:projects/Dataset/Security/CICIoT2023 --copy-links --progress --transfers 4 --checkers 8 --drive-chunk-size 256M --stats 30s --exclude ._* --exclude .DS_Store --exclude *.part
Transferred:   	   28.559 MiB / 12.811 GiB, 0%, 1.193 MiB/s, ETA 3h2m50s
Checks:                 0 / 0, -, Listed 190
Transferred:           15 / 184, 8%
Elapsed time:        29.9s
Transferring:
 * csv/CICIoT2023/part-00…-4d5862db7cb9-c000.csv: 31% / 67.079 MiB, 1.387 MiB/
 * csv/CICIoT2023/part-00…-4d5862db7cb9-c000.csv:  9% / 61.494 MiB, 474.769 Ki
 * csv/CICIoT2023/part-00…-4d5862db7cb9-c000.csv:  0% / 77.341 MiB, 19.999 KiB
 * csv/CICIoT2023/part-00…-4d5862db7cb9-c000.csv:  0% / 64.899 MiB, 0 B/s, -Transferred:   	   95.555 MiB / 12.811 GiB, 1%, 2.222 MiB/s, ETA 1h37m40s
Checks:                 0 / 0, -, Listed 190
Transferred:           15 / 184, 8%
Elapsed time:        59.9s
Transferring:
 * csv/CICIoT2023/par

0

In [3]:
verify_cmd = [RCLONE, "size", DRIVE_DESTINATION, "--exclude", "._*", "--exclude", ".DS_Store"]
print(" ".join(verify_cmd))
subprocess.check_call(verify_cmd)

/opt/homebrew/bin/rclone size gdrive:projects/Dataset/Security/CICIoT2023 --exclude ._* --exclude .DS_Store
Total objects: 184
Total size: 12.811 GiB (13755611744 Byte)


0